# 🏎️ F1 BI — SQL Avanzado
## Proyecto Final · Módulo 4: Inteligencia de Negocios y SQL Avanzado

Este notebook ejecuta las **5 técnicas de SQL avanzado** directamente sobre el modelo dimensional en Aurora PostgreSQL y muestra los resultados como DataFrames en Colab.

| # | Técnica | Pregunta analítica |
|:---:|---|---|
| 1 | **Window Functions** — `SUM() OVER`, `LAG()`, `RANK()` | ¿Quién lidera el campeonato en cada ronda de la temporada 2023? |
| 2 | **CTEs anidados** — 3 niveles | ¿Qué constructores son más eficientes y tienen menor tasa de falla mecánica por era? |
| 3 | **`PERCENTILE_CONT`** — p25, p50, p75 | ¿Desde qué posición mediana se gana en cada circuito? |
| 4 | **Funciones de fecha** — `DATE_PART`, `AGE` + `LAG()` | ¿Cómo evoluciona la edad promedio de los ganadores por temporada? |
| 5 | **Stored Procedure** + consulta directa | Resumen de dominancia por constructor para cualquier temporada |

## 0. Setup — Conexión a Aurora PostgreSQL

In [ ]:
!pip install sqlalchemy psycopg2-binary pandas tabulate -q

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from IPython.display import display, HTML
from google.colab import userdata

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.1f}'.format)

# Conexión Aurora — base de datos: postgres, schema: f1_dw
AURORA_HOST     = userdata.get('F1_HOST')
AURORA_PASSWORD = userdata.get('F1_data')
AURORA_USER     = userdata.get('AURORA_USER')

engine = create_engine(
    f'postgresql+psycopg2://{AURORA_USER}:{AURORA_PASSWORD}@{AURORA_HOST}:5432/postgres',
    pool_pre_ping=True
)

with engine.connect() as conn:
    db  = conn.execute(text('SELECT current_database()')).scalar()
    n   = conn.execute(text("SELECT COUNT(*) FROM f1_dw.fact_resultado_carrera")).scalar()
    print(f'Conectado a  : {db}')
    print(f'Filas en fact: {n:,}')

## Técnica 1 — Window Functions

**Pregunta:** ¿Cuál es el ranking acumulado de puntos por piloto en la temporada 2023 y quién lidera el campeonato en cada ronda?

### ¿Qué hace este código?

Usa **3 CTEs encadenados** para evitar anidar window functions (lo que PostgreSQL no permite):

- **`puntos_por_ronda`** — extrae los puntos de cada piloto por carrera en 2023
- **`acumulado`** — calcula dos window functions por separado:
  - `SUM() OVER (... ORDER BY numero_ronda)` → puntos acumulados hasta esa ronda
  - `LAG(puntos, 1, 0)` → puntos obtenidos en la ronda anterior (para ver tendencia)
- **`con_ranking`** — aplica `RANK() OVER` sobre `puntos_acumulados` ya calculado

El resultado final muestra el **Top 5 del campeonato en cada ronda** con su variación respecto a la carrera anterior.

In [ ]:
sql_t1 = """
SET search_path TO f1_dw;

WITH puntos_por_ronda AS (
    SELECT
        t.temporada,
        t.numero_ronda,
        t.nombre_gp,
        p.nombre_completo                                   AS piloto,
        c.nombre                                            AS constructor,
        f.puntos
    FROM fact_resultado_carrera f
    JOIN dim_piloto      p ON f.piloto_sk      = p.piloto_sk
    JOIN dim_constructor c ON f.constructor_sk = c.constructor_sk
    JOIN dim_tiempo      t ON f.tiempo_sk      = t.tiempo_sk
    WHERE t.temporada = 2023
),
acumulado AS (
    SELECT
        temporada, numero_ronda, nombre_gp, piloto, constructor, puntos,
        SUM(puntos) OVER (
            PARTITION BY temporada, piloto
            ORDER BY numero_ronda
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )                                                   AS puntos_acumulados,
        LAG(puntos, 1, 0) OVER (
            PARTITION BY temporada, piloto
            ORDER BY numero_ronda
        )                                                   AS puntos_ronda_anterior
    FROM puntos_por_ronda
),
con_ranking AS (
    SELECT
        temporada, numero_ronda, nombre_gp, piloto, constructor,
        puntos, puntos_acumulados, puntos_ronda_anterior,
        RANK() OVER (
            PARTITION BY temporada, numero_ronda
            ORDER BY puntos_acumulados DESC
        )                                                   AS posicion_campeonato
    FROM acumulado
)
SELECT
    numero_ronda,
    nombre_gp,
    piloto,
    constructor,
    puntos                                                  AS puntos_ronda,
    puntos_acumulados,
    posicion_campeonato,
    puntos - puntos_ronda_anterior                          AS variacion_vs_anterior
FROM con_ranking
WHERE posicion_campeonato <= 5
ORDER BY numero_ronda, posicion_campeonato;
"""

with engine.connect() as conn:
    df_t1 = pd.read_sql(text(sql_t1), conn)

print(f'Filas resultado: {len(df_t1):,}')
display(df_t1.head(25))

## Técnica 2 — CTEs Anidados (3 niveles)

**Pregunta:** ¿Qué constructores tienen mayor eficiencia (posición de salida vs llegada) y menor tasa de abandono mecánico por era histórica?

### ¿Qué hace este código?

Construye la consulta en **3 niveles de CTEs**, donde cada nivel usa el resultado del anterior:

- **`base`** (nivel 1) — hace los JOINs con dimensiones y filtra grids válidos (> 0). Es la fuente limpia de datos para los siguientes niveles.
- **`metricas_constructor`** (nivel 2) — agrega por constructor y era calculando:
  - `pct_finalizados`: % de veces que terminó la carrera
  - `pct_abandono_mecanico`: % de abandonos por falla técnica
  - `avg_delta_posicion`: posiciones ganadas o perdidas en promedio
  - `NULLIF(COUNT(*), 0)` evita división por cero
- **`ranking_eficiencia`** (nivel 3) — aplica `RANK() OVER (PARTITION BY era_f1)` sobre las métricas ya calculadas. El ranking es independiente por era.

El filtro final `rank_eficiencia_era <= 3` muestra el **Top 3 más eficiente por era**.

In [ ]:
sql_t2 = """
SET search_path TO f1_dw;

WITH base AS (
    SELECT
        c.nombre                                            AS constructor,
        c.era_f1,
        f.posicion_salida,
        f.posicion_final,
        f.delta_posicion,
        f.es_abandono,
        e.categoria                                         AS categoria_estado
    FROM fact_resultado_carrera f
    JOIN dim_constructor c ON f.constructor_sk = c.constructor_sk
    JOIN dim_estado      e ON f.estado_sk      = e.estado_sk
    WHERE f.posicion_salida IS NOT NULL
      AND f.posicion_salida > 0
),
metricas_constructor AS (
    SELECT
        constructor,
        era_f1,
        COUNT(*)                                            AS total_entradas,
        ROUND(AVG(delta_posicion)::NUMERIC, 2)              AS avg_delta_posicion,
        ROUND(
            SUM(CASE WHEN NOT es_abandono THEN 1 ELSE 0 END)::NUMERIC
            / NULLIF(COUNT(*), 0) * 100
        , 1)                                                AS pct_finalizados,
        ROUND(
            SUM(CASE WHEN categoria_estado = 'Abandono mecanico'
                THEN 1 ELSE 0 END)::NUMERIC
            / NULLIF(COUNT(*), 0) * 100
        , 1)                                                AS pct_abandono_mecanico,
        SUM(CASE WHEN posicion_final = 1 THEN 1 ELSE 0 END) AS victorias
    FROM base
    GROUP BY constructor, era_f1
    HAVING COUNT(*) >= 20
),
ranking_eficiencia AS (
    SELECT
        constructor, era_f1, total_entradas,
        avg_delta_posicion, pct_finalizados,
        pct_abandono_mecanico, victorias,
        RANK() OVER (
            PARTITION BY era_f1
            ORDER BY pct_finalizados DESC, avg_delta_posicion DESC
        )                                                   AS rank_eficiencia_era
    FROM metricas_constructor
)
SELECT
    era_f1, rank_eficiencia_era AS rank, constructor,
    total_entradas, avg_delta_posicion,
    pct_finalizados AS pct_termina,
    pct_abandono_mecanico AS pct_falla_mecanica,
    victorias
FROM ranking_eficiencia
WHERE rank_eficiencia_era <= 3
ORDER BY era_f1, rank_eficiencia_era;
"""

with engine.connect() as conn:
    df_t2 = pd.read_sql(text(sql_t2), conn)

print(f'Filas resultado: {len(df_t2):,}')
display(df_t2)

## Técnica 3 — PERCENTILE_CONT

**Pregunta:** ¿Desde qué posición de salida mediana se gana en cada circuito y qué tan dispersos están los resultados?

### ¿Qué hace este código?

`PERCENTILE_CONT` es una **función de orden de conjunto** que calcula percentiles continuos interpolando entre valores. A diferencia de `AVG`, no se distorsiona por outliers.

- **`PERCENTILE_CONT(0.5)`** → mediana de la posición de salida de los ganadores
- **`PERCENTILE_CONT(0.25)`** → percentil 25 (p25): posición desde donde gana el 25% inferior
- **`PERCENTILE_CONT(0.75)`** → percentil 75 (p75): posición desde donde gana el 75% inferior
- **`p75 - p25`** es el rango intercuartil (IQR): cuanto más pequeño, más predecible es el circuito

La cláusula `WITHIN GROUP (ORDER BY posicion_salida)` le indica a la función **sobre qué columna** calcular el percentil.

**Interpretación:** un circuito con `mediana_grid = 1` significa que históricamente se gana casi siempre desde la pole. Un circuito con `mediana_grid = 4` es más abierto a estrategias desde posiciones más atrás.

In [ ]:
sql_t3 = """
SET search_path TO f1_dw;

WITH ganadores AS (
    SELECT
        ci.nombre                                           AS circuito,
        ci.pais,
        f.posicion_salida
    FROM fact_resultado_carrera f
    JOIN dim_circuito ci ON f.circuito_sk = ci.circuito_sk
    WHERE f.posicion_final  = 1
      AND f.posicion_salida IS NOT NULL
      AND f.posicion_salida > 0
)
SELECT
    circuito,
    pais,
    COUNT(*)                                                AS total_ganadores,
    PERCENTILE_CONT(0.5)  WITHIN GROUP (ORDER BY posicion_salida) AS mediana_grid,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY posicion_salida) AS p25_grid,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY posicion_salida) AS p75_grid,
    ROUND(
        SUM(CASE WHEN posicion_salida = 1 THEN 1 ELSE 0 END)::NUMERIC
        / NULLIF(COUNT(*), 0) * 100
    , 1)                                                    AS pct_gana_desde_pole,
    MIN(posicion_salida)                                    AS mejor_grid_ganador,
    MAX(posicion_salida)                                    AS peor_grid_ganador
FROM ganadores
GROUP BY circuito, pais
HAVING COUNT(*) >= 5
ORDER BY mediana_grid ASC;
"""

with engine.connect() as conn:
    df_t3 = pd.read_sql(text(sql_t3), conn)

print(f'Circuitos analizados: {len(df_t3):,}')
print(f'\nTop 10 circuitos donde más importa la pole position:')
display(df_t3.head(10))
print(f'\nTop 10 circuitos más abiertos (ganadores desde más atrás):')
display(df_t3.tail(10))

## Técnica 4 — Funciones de Fecha + Window Functions

**Pregunta:** ¿Cómo evoluciona la edad promedio de los ganadores de carrera por temporada? ¿La F1 moderna premia a pilotos más jóvenes?

### ¿Qué hace este código?

Combina **funciones de fecha** con **window functions** en 2 CTEs:

- **`edad_ganadores`** (CTE 1):
  - `AGE(fecha_carrera, fecha_nacimiento)` → calcula el intervalo exacto entre dos fechas
  - `DATE_PART('year', AGE(...))` → extrae solo los años de ese intervalo
  - Resultado: la edad exacta del piloto el día que ganó la carrera

- **`agregado_temporada`** (CTE 2):
  - Materializa `avg_edad` por temporada antes de aplicar `LAG()`
  - Esto es necesario porque PostgreSQL no permite aplicar `LAG()` directamente
    sobre una expresión agregada (`AVG()`) en el mismo `SELECT`

- **`SELECT` final**:
  - `LAG(avg_edad) OVER (ORDER BY temporada)` → edad promedio de la temporada anterior
  - `avg_edad - LAG(avg_edad)` → variación: positivo = ganadores más viejos ese año

**Interpretación:** valores negativos en `variacion_edad_vs_anterior` indican temporadas donde los ganadores fueron más jóvenes que el año previo.

In [ ]:
sql_t4 = """
SET search_path TO f1_dw;

WITH edad_ganadores AS (
    SELECT
        t.temporada,
        t.era_f1,
        DATE_PART('year', AGE(t.fecha, p.fecha_nacimiento)) AS edad_en_carrera
    FROM fact_resultado_carrera f
    JOIN dim_piloto p ON f.piloto_sk = p.piloto_sk
    JOIN dim_tiempo t ON f.tiempo_sk = t.tiempo_sk
    WHERE f.posicion_final      = 1
      AND p.fecha_nacimiento IS NOT NULL
      AND t.fecha            IS NOT NULL
),
agregado_temporada AS (
    SELECT
        era_f1,
        temporada,
        COUNT(*)                                            AS carreras_ganadas,
        ROUND(AVG(edad_en_carrera)::NUMERIC, 1)             AS avg_edad,
        MIN(edad_en_carrera)                                AS edad_minima,
        MAX(edad_en_carrera)                                AS edad_maxima,
        PERCENTILE_CONT(0.5) WITHIN GROUP (
            ORDER BY edad_en_carrera
        )                                                   AS mediana_edad
    FROM edad_ganadores
    GROUP BY era_f1, temporada
)
SELECT
    era_f1,
    temporada,
    carreras_ganadas,
    avg_edad,
    edad_minima,
    edad_maxima,
    mediana_edad,
    avg_edad - LAG(avg_edad) OVER (
        ORDER BY temporada
    )                                                       AS variacion_edad_vs_anterior
FROM agregado_temporada
ORDER BY temporada;
"""

with engine.connect() as conn:
    df_t4 = pd.read_sql(text(sql_t4), conn)

print(f'Temporadas analizadas: {len(df_t4):,}')
display(df_t4)

## Técnica 5 — Stored Procedure + Consulta directa

**Propósito:** Crear un procedimiento reutilizable que genere el resumen de dominancia por constructor para cualquier temporada con un solo `CALL`.

### ¿Qué hace este código?

**Parte A — Crear el Stored Procedure:**
- `CREATE OR REPLACE PROCEDURE` define el procedimiento en el schema `f1_dw`
- `LANGUAGE plpgsql` indica que usa PL/pgSQL (lenguaje procedural de PostgreSQL)
- `DECLARE v_count INT` declara una variable local para verificar datos
- `SELECT COUNT(*) INTO v_count` carga el conteo en la variable
- `IF v_count = 0 THEN RAISE EXCEPTION` valida que la temporada tenga datos antes de continuar
- `DROP TABLE IF EXISTS + CREATE TEMP TABLE` crea una tabla temporal con los agregados
- `RAISE NOTICE` imprime mensajes de progreso en el log del servidor

**Parte B — Consulta directa equivalente:**
- Misma lógica pero ejecutable con `pd.read_sql()` directamente desde Colab
- `RANK() OVER (ORDER BY SUM(f.puntos) DESC)` rankea constructores por puntos totales
- Cambiar el año en el `WHERE` para analizar cualquier temporada

**¿Por qué el RANK() va en la consulta y no en la tabla temporal?**
PostgreSQL no permite usar window functions directamente dentro de `CREATE TABLE AS ... GROUP BY`. La solución es separar: la tabla temporal guarda los agregados simples, y el `RANK()` se aplica en la consulta final.

In [ ]:
# Parte A: Crear el Stored Procedure en Aurora
sql_create_proc = """
CREATE OR REPLACE PROCEDURE f1_dw.resumen_temporada(p_temporada INT)
LANGUAGE plpgsql
AS $$
DECLARE
    v_count INT;
BEGIN
    RAISE NOTICE '================================================';
    RAISE NOTICE 'Resumen Temporada: %', p_temporada;
    RAISE NOTICE '================================================';

    SELECT COUNT(*) INTO v_count
    FROM f1_dw.fact_resultado_carrera f
    JOIN f1_dw.dim_tiempo t ON f.tiempo_sk = t.tiempo_sk
    WHERE t.temporada = p_temporada;

    IF v_count = 0 THEN
        RAISE EXCEPTION 'No se encontraron datos para la temporada %', p_temporada;
    END IF;

    DROP TABLE IF EXISTS tmp_resumen;
    CREATE TEMP TABLE tmp_resumen AS
    SELECT
        c.nombre                                            AS constructor,
        c.era_f1,
        COUNT(*)                                            AS entradas,
        SUM(f.puntos)                                       AS puntos_totales,
        SUM(CASE WHEN f.posicion_final = 1  THEN 1 ELSE 0 END) AS victorias,
        SUM(CASE WHEN f.posicion_final <= 3 THEN 1 ELSE 0 END) AS podios,
        ROUND(AVG(f.posicion_final)::NUMERIC, 1)            AS avg_pos_final,
        SUM(CASE WHEN f.es_abandono THEN 1 ELSE 0 END)      AS abandonos
    FROM f1_dw.fact_resultado_carrera f
    JOIN f1_dw.dim_constructor c ON f.constructor_sk = c.constructor_sk
    JOIN f1_dw.dim_tiempo      t ON f.tiempo_sk      = t.tiempo_sk
    WHERE t.temporada = p_temporada
    GROUP BY c.nombre, c.era_f1;

    RAISE NOTICE 'Datos cargados: % constructores en temporada %',
        (SELECT COUNT(*) FROM tmp_resumen), p_temporada;
END;
$$;
"""

with engine.begin() as conn:
    conn.execute(text(sql_create_proc))
print('✅ Stored Procedure f1_dw.resumen_temporada creado correctamente')

In [ ]:
# Parte B: Consulta directa equivalente — cambia el año para cualquier temporada
TEMPORADA = 2023   # <-- Modifica aquí para analizar otra temporada

sql_t5 = f"""
SET search_path TO f1_dw;

SELECT
    c.nombre                                                AS constructor,
    c.era_f1,
    COUNT(*)                                                AS entradas,
    SUM(f.puntos)                                           AS puntos_totales,
    SUM(CASE WHEN f.posicion_final = 1  THEN 1 ELSE 0 END)  AS victorias,
    SUM(CASE WHEN f.posicion_final <= 3 THEN 1 ELSE 0 END)  AS podios,
    ROUND(AVG(f.posicion_final)::NUMERIC, 1)                AS avg_pos_final,
    SUM(CASE WHEN f.es_abandono THEN 1 ELSE 0 END)          AS abandonos,
    RANK() OVER (ORDER BY SUM(f.puntos) DESC)               AS posicion_campeonato
FROM fact_resultado_carrera f
JOIN dim_constructor c ON f.constructor_sk = c.constructor_sk
JOIN dim_tiempo      t ON f.tiempo_sk      = t.tiempo_sk
WHERE t.temporada = {TEMPORADA}
GROUP BY c.nombre, c.era_f1
ORDER BY posicion_campeonato;
"""

with engine.connect() as conn:
    df_t5 = pd.read_sql(text(sql_t5), conn)

print(f'Temporada {TEMPORADA} — {len(df_t5)} constructores')
display(df_t5)

## Resumen — Técnicas de SQL avanzado aplicadas

| Técnica | Funciones usadas | Pregunta resuelta |
|---|---|---|
| **Window Functions** | `SUM() OVER`, `LAG()`, `RANK() OVER` | Ranking del campeonato por ronda |
| **CTEs anidados** | `WITH` en 3 niveles | Eficiencia de constructores por era |
| **PERCENTILE_CONT** | `WITHIN GROUP (ORDER BY)` | Posición mediana de ganadores por circuito |
| **Funciones de fecha** | `DATE_PART()`, `AGE()` + `LAG()` | Evolución de edad de ganadores |
| **Stored Procedure** | `CREATE PROCEDURE`, `plpgsql` | Resumen reutilizable por temporada |

> Todas las consultas resuelven preguntas reales de la pregunta analítica del proyecto, no son ejercicios sintéticos.

In [ ]:
# Verificación final: shapes de todos los resultados
print('Resultados obtenidos por técnica:')
print(f'  Técnica 1 (Window Functions) : {len(df_t1):>6,} filas')
print(f'  Técnica 2 (CTEs anidados)    : {len(df_t2):>6,} filas')
print(f'  Técnica 3 (PERCENTILE_CONT)  : {len(df_t3):>6,} filas')
print(f'  Técnica 4 (Funciones fecha)  : {len(df_t4):>6,} filas')
print(f'  Técnica 5 (Stored Procedure) : {len(df_t5):>6,} filas')